# Real-time Audio Conditioning Demo with Magenta RT

This notebook demonstrates how to condition music generation in Magenta RT based on user-provided audio snippets. You'll be able to upload short audio clips and use their musical style to influence the generated music interactively.

**Note on "Real-time":** In this Colab environment, "real-time" interaction means the model will react to audio snippets you upload during designated pauses in the generation process, rather than continuous microphone input.

**We will cover:**
1. Setting up the environment and loading models.
2. Understanding how Magenta RT can use audio for style conditioning via MusicCoCa.
3. Uploading your own audio snippets.
4. Setting an initial musical style (optional, with a text prompt).
5. Interactively generating music, injecting the style of your audio snippets at chosen moments.
6. Listening to the final co-created musical piece.

## 1. Setup: Install Dependencies, Import Libraries, and Load Models

In [ ]:
# @title Install Magenta RT and dependencies (may take ~5 minutes)
# @markdown **Important:** Make sure you are using a TPU runtime for this notebook (`Runtime > Change runtime type > TPU`). Colab may prompt you to restart the session after installation. If so, wait for the cell to finish, then restart and run subsequent cells.

print("Updating package list and installing ffmpeg for wider audio format support...")
!apt-get update -qq && apt-get install -y ffmpeg

print("Installing Magenta RT and dependencies...")
# Clone the repository (if not already done or running in a fresh environment)
!git clone https://github.com/magenta/magenta-realtime.git

# Uninstall existing TensorFlow versions to avoid conflicts and install specific nightly versions
_all_tf = 'tensorflow tf-nightly tensorflow-cpu tf-nightly-cpu tensorflow-tpu tf-nightly-tpu tensorflow-hub tf-hub-nightly tensorflow-text tensorflow-text-nightly'
_nightly_tf = 'tf-nightly tensorflow-text-nightly tf-hub-nightly'

# Install Magenta RT with TPU support
!pip install -e magenta-realtime/[tpu] && pip uninstall -y {_all_tf} && pip install {_nightly_tf}
print("Installation complete. Please restart the session if prompted by Colab.")

In [ ]:
# @title Import libraries and initialize models (may take ~5 minutes after restart)
# @markdown If you restarted the session after the previous cell, run this cell.

import numpy as np
from IPython.display import Audio, display
from google.colab import files
import os
import time
import math

try:
    from magenta_rt import system, audio, musiccoca
    print("Libraries imported successfully.")
except ImportError as e:
    print(f"ImportError: {e}. This might be because the session hasn't been restarted after installation. Please restart and try again.")

MRT = None
STYLE_MODEL = None

try:
    print("\nInitializing MagentaRT model (MRT)...")
    MRT = system.MagentaRT(
        tag="large", device="tpu:v2-8", skip_cache=True, lazy=False
    )
    print("MagentaRT model initialized successfully.")

    print("\nInitializing MusicCoCa style model...")
    STYLE_MODEL = musiccoca.MusicCoCa()
    print("MusicCoCa model initialized successfully.")

except NameError:
    print("NameError: MagentaRT or MusicCoCa class not found. This usually means the session needs a restart after installation.")
    print("Please ensure the installation cell completed, restart the session (`Runtime > Restart session`), and then re-run this cell.")
except Exception as e:
    print(f"An error occurred during model initialization: {e}")
    print("Please ensure you are on a TPU runtime and have restarted the session if prompted after installation.")

**Troubleshooting:** If the cell above reports `NameError` or `ImportError`, it's likely because Colab needs a session restart after the `pip install` command in the first code cell. 
1. Make sure the first code cell (installation) has finished running.
2. Restart the Colab session: `Runtime > Restart session`.
3. Re-run the cell above (Import libraries and initialize models).

Let's add a check to ensure models are loaded before proceeding:

In [ ]:
# @title Confirm Models are Loaded
if MRT is None or STYLE_MODEL is None:
    print("Models not loaded. Please re-run the previous cell (Import libraries and initialize models).")
    print("Ensure you have restarted the session after installation if you encountered errors.")
    # Attempt to load again as a fallback, in case the user runs this cell directly after restart
    try:
        if MRT is None:
            print("Attempting to load MagentaRT model again...")
            from magenta_rt import system
            MRT = system.MagentaRT(tag="large", device="tpu:v2-8", skip_cache=True, lazy=False)
            print("MagentaRT model loaded.")
        if STYLE_MODEL is None:
            print("Attempting to load MusicCoCa model again...")
            from magenta_rt import musiccoca
            STYLE_MODEL = musiccoca.MusicCoCa()
            print("MusicCoCa model loaded.")
    except Exception as e:
        print(f"Fallback model loading failed: {e}")
else:
    print("MagentaRT (MRT) and MusicCoCa (STYLE_MODEL) are loaded and ready!")

## 2. Understanding Audio Conditioning

Magenta RT can generate music conditioned on various styles. While text prompts are common, we can also use **MusicCoCa** to extract a style embedding from an audio clip. This embedding captures musical characteristics of the audio, which can then be used to guide Magenta RT's generation process.

In this notebook, you will upload short audio snippets. We will then:
1. Load the audio snippet.
2. Use `STYLE_MODEL.embed()` (our MusicCoCa instance) to get its style embedding.
3. This audio style embedding can then be blended with an optional text prompt embedding or used directly to influence the `style` parameter of `MRT.generate_chunk()`.

## 3. Step 1: Upload Your Audio Snippet(s)

Use the cell below to upload one or more short audio files (e.g., WAV or MP3 format, 2-10 seconds long is recommended). These snippets will be used to condition the music generation.

In [ ]:
# @title Upload audio files
UPLOADED_SNIPPETS_DIR = 'uploaded_snippets'
if not os.path.exists(UPLOADED_SNIPPETS_DIR):
    os.makedirs(UPLOADED_SNIPPETS_DIR)

uploaded_files = files.upload() # This will open a file dialog

saved_snippets = {}
if uploaded_files:
    print("\nProcessing uploaded files...")
    for filename, content in uploaded_files.items():
        filepath = os.path.join(UPLOADED_SNIPPETS_DIR, filename)
        with open(filepath, 'wb') as f:
            f.write(content)
        print(f"Saved '{filename}' to '{filepath}'")
        try:
            # Load into Waveform object to verify and store
            # Explicitly use magenta_rt.audio to ensure it's found
            waveform = magenta_rt.audio.Waveform.from_file(filepath)
            saved_snippets[filename] = {
                'filepath': filepath,
                'waveform': waveform,
                'duration': waveform.duration_seconds
            }
            print(f"  Successfully loaded '{filename}' ({waveform.duration_seconds:.2f}s, {waveform.sample_rate}Hz, {waveform.num_channels} channels)")
        except Exception as e:
            print(f"  Error loading '{filename}' as audio: {e}. Please ensure it's a valid audio file.")
    print("\nAvailable audio snippets:")
    if saved_snippets:
        for i, (name, data) in enumerate(saved_snippets.items()):
            print(f"  {i+1}. {name} ({data['duration']:.2f}s)")
    else:
        print("  No valid audio snippets loaded.")
else:
    print("No files were uploaded.")

If you want to add more snippets later, you can re-run the cell above.

In [ ]:
# @title Helper function to get style embedding from an audio snippet
def get_audio_snippet_embedding(snippet_name: str):
    """Gets the MusicCoCa style embedding for a named uploaded snippet."""
    if not STYLE_MODEL:
        print("MusicCoCa model (STYLE_MODEL) not loaded. Cannot get embedding.")
        return None
    if snippet_name not in saved_snippets:
        print(f"Snippet '{snippet_name}' not found in saved snippets.")
        return None

    try:
        print(f"Calculating embedding for '{snippet_name}'...")
        waveform = saved_snippets[snippet_name]['waveform']
        # MusicCoCa expects a batch, so we add a dimension and then take the first result
        embedding = STYLE_MODEL.embed(waveform)[0]
        print(f"Successfully got embedding for '{snippet_name}'.")
        return embedding
    except Exception as e:
        print(f"Error generating embedding for '{snippet_name}': {e}")
        return None

# Example usage (optional - to test if a snippet embeds correctly):
# if saved_snippets:
#     first_snippet_name = list(saved_snippets.keys())[0]
#     test_embedding = get_audio_snippet_embedding(first_snippet_name)
#     if test_embedding is not None:
#         print(f"Test embedding shape: {test_embedding.shape}")

## 4. Step 2: Set Initial Generation Style (Optional)

You can start with a text prompt to set an initial musical direction. If you leave this blank, the generation will be primarily influenced by the first audio snippet you choose to inject, or it might start from a more neutral/random state if no audio snippet is injected immediately.

In [ ]:
# @title Define an initial text prompt (optional)
initial_text_prompt = "chill electronic beat"  # @param {type:"string"}

current_text_embedding = None
if initial_text_prompt and initial_text_prompt.strip() and MRT:
    print(f"Embedding initial text prompt: '{initial_text_prompt}'")
    current_text_embedding = MRT.embed_style(initial_text_prompt)
    print("Initial text prompt embedded.")
else:
    print("No initial text prompt provided or MRT model not loaded. Will rely on audio snippets or default generation.")

## 5. Step 3: Interactive Generation with Audio Snippet Injection

This is the main interactive loop. Music will be generated in segments. After each segment, you'll be prompted to decide how to influence the next part of the music.

**Generation Parameters:**
- `chunks_per_segment`: How many 2-second chunks are generated before pausing for your input.
- `total_segments_to_generate`: How many interactive segments to run.
- `audio_snippet_influence`: When an audio snippet is injected, this weight (0.0 to 1.0) determines its influence relative to the ongoing text prompt style. 1.0 means the audio snippet style fully replaces the text prompt style for that injection.

In [ ]:
# @title Interactive Generation Loop

chunks_per_segment = 5  # @param {type:"integer"} (e.g., 5 chunks = 10 seconds of audio)
total_segments_to_generate = 10 # @param {type:"integer"} (e.g., 10 segments * 10s/segment = 100s total)
audio_snippet_influence = 0.7 # @param {type:"slider", min:0, max:1, step:0.1}

all_generated_chunks_interactive = []
current_model_state_interactive = None
active_style_embedding = current_text_embedding if current_text_embedding is not None else np.zeros((768,), dtype=np.float32) # Default to zeros if no text

CHUNK_LENGTH_SECONDS_MRT = MRT.config.chunk_length_seconds if MRT else 2.0

print(f"Starting interactive generation. Each segment will be {chunks_per_segment * CHUNK_LENGTH_SECONDS_MRT} seconds.")
print("After each segment, you'll be prompted for input.")

for segment_num in range(total_segments_to_generate):
    print(f"\n--- Generating Segment {segment_num + 1}/{total_segments_to_generate} ---")
    segment_audio_chunks = []
    for chunk_num_in_segment in range(chunks_per_segment):
        if MRT is None:
            print("MRT model not loaded. Aborting generation.")
            break
        
        # Use a unique seed for each chunk for variety, can be fixed if desired
        seed = int(time.time() * 1000) % (2**32 -1) 

        generated_chunk, current_model_state_interactive = MRT.generate_chunk(
            state=current_model_state_interactive,
            style=active_style_embedding,
            seed=seed,
            temperature=1.0, # Adjust as desired
            top_k=0,         # Adjust as desired
            guidance_weight=3.0 # Adjust as desired
        )
        segment_audio_chunks.append(generated_chunk)
        all_generated_chunks_interactive.append(generated_chunk)
        print(f"  Generated chunk {chunk_num_in_segment + 1}/{chunks_per_segment} for segment {segment_num + 1}", end='\r')
    
    if MRT is None: break # Break outer loop if model disappeared
    print(f"\nSegment {segment_num + 1} generated.")

    # --- User Interaction Part ---
    if segment_num < total_segments_to_generate - 1: # Don't ask for input after the last segment
        print("\nWhat would you like to do for the next segment?")
        print("  1. Continue with current style.")
        print("  2. Inject an audio snippet style.")
        print("  3. Change text prompt.")
        print("  4. End generation.")
        
        while True:
            try:
                choice = input("Enter your choice (1-4): ").strip()
                if choice in ['1', '2', '3', '4']:
                    break
                else:
                    print("Invalid choice. Please enter a number between 1 and 4.")
            except EOFError:
                print("Input stream closed, ending generation.")
                choice = '4'
                break

        if choice == '1': # Continue
            if initial_text_prompt and current_text_embedding is not None:
                print(f"Continuing with style from text prompt: '{initial_text_prompt}'.")
            elif not np.all(active_style_embedding == 0):
                print("Continuing with the previously active audio snippet style.")
            else:
                print("Continuing with a neutral/default style.")
            # active_style_embedding remains the same

        elif choice == '2': # Inject audio snippet
            if not saved_snippets:
                print("No audio snippets uploaded. Continuing with current style.")
                continue
            print("Available audio snippets:")
            snippet_names = list(saved_snippets.keys())
            for i, name in enumerate(snippet_names):
                print(f"  {i+1}. {name} ({saved_snippets[name]['duration']:.2f}s)")
            
            while True:
                try:
                    snippet_choice_idx = input(f"Enter snippet number to inject (1-{len(snippet_names)}): ").strip()
                    snippet_choice_idx = int(snippet_choice_idx) - 1
                    if 0 <= snippet_choice_idx < len(snippet_names):
                        chosen_snippet_name = snippet_names[snippet_choice_idx]
                        print(f"Injecting style from '{chosen_snippet_name}'...")
                        audio_embed = get_audio_snippet_embedding(chosen_snippet_name)
                        if audio_embed is not None:
                            if current_text_embedding is not None:
                                # Blend audio with text embedding
                                active_style_embedding = (
                                    (1.0 - audio_snippet_influence) * current_text_embedding + 
                                    audio_snippet_influence * audio_embed
                                )
                                print(f"Blended text prompt with '{chosen_snippet_name}' (influence: {audio_snippet_influence*100}%). ")
                            else:
                                # Use audio embedding directly
                                active_style_embedding = audio_embed
                                print(f"Using style from '{chosen_snippet_name}' directly.")
                        else:
                            print(f"Could not get embedding for '{chosen_snippet_name}'. Continuing with previous style.")
                        break
                    else:
                        print("Invalid snippet number.")
                except ValueError:
                    print("Invalid input. Please enter a number.")
                except EOFError:
                    print("Input stream closed, defaulting to continue with current style.")
                    break

        elif choice == '3': # Change text prompt
            try:
                new_text_prompt = input("Enter new text prompt (or leave blank to remove text influence): ").strip()
                if new_text_prompt and MRT:
                    print(f"Embedding new text prompt: '{new_text_prompt}'")
                    current_text_embedding = MRT.embed_style(new_text_prompt)
                    active_style_embedding = current_text_embedding
                    initial_text_prompt = new_text_prompt # Update for future reference
                    print("New text prompt embedded and activated.")
                else:
                    current_text_embedding = None
                    active_style_embedding = np.zeros((768,), dtype=np.float32) # Reset to neutral if no text
                    initial_text_prompt = ""
                    print("Text prompt influence removed or model not loaded.")
            except EOFError:
                print("Input stream closed, continuing with current style.")

        elif choice == '4': # End generation
            print("Ending interactive generation.")
            break
    elif segment_num == total_segments_to_generate -1:
        print("\nFinished all planned segments.")

print("\n--- Interactive generation complete ---")


## 6. Step 4: Concatenate and Play Final Audio

Once the interactive generation is complete, this cell will concatenate all the generated audio chunks and provide an audio player to listen to your creation.

In [ ]:
# @title Concatenate all generated chunks and display audio

if MRT is None:
    print("MagentaRT model (MRT) is not loaded. Cannot process audio.")
elif all_generated_chunks_interactive:
    print(f"\nConcatenating {len(all_generated_chunks_interactive)} audio chunks...")
    # Use the crossfade time from the MRT model's config
    crossfade_duration_seconds = MRT.config.crossfade_length 

    final_interactive_audio = audio.concatenate(
        all_generated_chunks_interactive,
        crossfade_time=crossfade_duration_seconds
    )
    print("Concatenation complete.")

    actual_duration_seconds = final_interactive_audio.duration_seconds
    print(f"Final audio duration: {actual_duration_seconds // 60:.0f} minutes and {actual_duration_seconds % 60:.2f} seconds.")
    display(Audio(final_interactive_audio.samples.T, rate=final_interactive_audio.sample_rate))
else:
    print("No audio chunks were generated in the interactive session.")

## 7. Experimentation Ideas

Now that you've gone through the process, here are a few ideas to explore further:

- **Diverse Snippets:** Try uploading various types of audio snippets:
    - Short melodic phrases (e.g., a sung note, a synth line).
    - Rhythmic patterns (e.g., a drum beat, a percussive loop).
    - Textural sounds (e.g., ambient noise, a sustained chord).
    Observe how different types of snippets influence the generation.
- **Blending Influence:** Adjust the `audio_snippet_influence` slider in the interactive loop. 
    - A lower value will make the audio snippet's style more subtle, blending more with the base text prompt.
    - A higher value will let the audio snippet dominate the style during its injection.
- **Text Prompt Interaction:** 
    - Start with no text prompt and let the first audio snippet define the entire style.
    - Try very contrasting text prompts and audio snippets to see how they merge (or clash!).
    - Change the text prompt mid-way and see how it interacts with subsequent audio snippet injections.
- **Segment Length:** Modify `chunks_per_segment`. Shorter segments allow for more frequent interaction, while longer segments let a style develop more before you intervene.
- **Generation Parameters:** In the `MRT.generate_chunk` call within the interactive loop, experiment with `temperature`, `top_k`, and `guidance_weight` to alter the creativity and adherence of the model.

## 8. Conclusion and Limitations

This notebook demonstrated how to use Magenta RT with MusicCoCa to achieve a form of real-time audio conditioning by injecting the style of uploaded audio snippets during an interactive generation process. This allows for a dynamic and co-creative music generation experience where you can guide the model's output using your own sounds.

**Key Limitations to Keep in Mind:**
- **Snippet-Based, Not True Real-time Mic Input:** As mentioned, this Colab environment facilitates interaction through uploaded snippets at discrete points, not continuous, low-latency microphone input that directly shapes the waveform as it's generated.
- **Embedding Quality:** The effectiveness of the conditioning depends on the quality of the style embedding from MusicCoCa. Some audio snippets might translate into more distinct or influential style embeddings than others.
- **Blending Strategy:** The current blending is a simple weighted average. More sophisticated methods for combining text and audio style embeddings, or for transitioning between them, could be explored for different effects.

Happy experimenting!